# 🎯 Mock Assessment v2 — Friday Mixed Drill
**Date:** March 6, 2026  
**Time limit:** 30 minutes total  
**Rules:** No peeking at docs for at least 10 min per section. Struggle first.

---

## Setup
Run the cell below, then **start your timer**. ⏱️

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load datasets
customers = pd.read_csv('../data/customers.csv')
orders = pd.read_csv('../data/orders.csv')
surveys = pd.read_csv('../data/surveys.csv')

print(f"Customers: {customers.shape}")
print(f"Orders: {orders.shape}")
print(f"Surveys: {surveys.shape}")
print("\n✅ Data loaded. START YOUR TIMER NOW.")

Customers: (5000, 11)
Orders: (15000, 5)
Surveys: (3000, 4)

✅ Data loaded. START YOUR TIMER NOW.


---
## Part 1: Data Cleaning & Pandas (10 min)

**Scenario:** You're preparing the customer dataset for a churn prediction model.

### Q1.1
Find all columns with missing values and their missing percentages. Display them sorted descending by % missing.

In [ ]:
n_rows, n_columns = customers.shape

columns_with_na = {}
for column in customers.columns:
    n_missing = customers[column].isna().sum()
    if n_missing > 0:
        columns_with_na[column] = n_missing/n_rows
for k, v in sorted(columns_with_na.items(), key=lambda item: item[1], reverse=True):
    print(k,v)

# I mean, this works... This demonstrate a good grasp of how components work, but try to use more specialized tools.

gender 0.125
plan_type 0.1092
city 0.0468
annual_income 0.04
monthly_spend 0.016
age 0.003
signup_date 0.0024


In [ ]:
pct_change = customers.isna().mean().sort_values(ascending=False); pct_change[pct_change > 0]  # This is actually brilliant because the mean of booleans gives you the percentage of True values

gender           0.1250
plan_type        0.1092
city             0.0468
annual_income    0.0400
monthly_spend    0.0160
age              0.0030
signup_date      0.0024
dtype: float64

### Q1.2
Create a new column `customer_segment` based on `annual_income`:
- `'low'` if income < 25,000
- `'mid'` if income >= 25,000 and < 60,000
- `'high'` if income >= 60,000
- `NaN` if income is missing

Then show the **churn rate** (mean of `churned`) for each segment.

In [ ]:
# Your code here
conditions = [
    (customers["annual_income"] < 25_000),
    (customers["annual_income"] >= 25_000) & (customers["annual_income"] < 60_000),  # Don't forget to add proper upper, and lower boundries. No overlap!
    (customers["annual_income"] >= 60_000)
]
values = [
    "low",
    "mid",
    "high"
]
customers["customer_segment"] = np.select(conditions, values, default="NaN")

### Q1.3
Merge `customers` with `orders`. Compute each customer's **total spend** (sum of `amount`) and **order count**. Customers with no orders should show 0 for both.

Store the result in `customer_orders` and show the top 5 spenders.

In [84]:
# Your code here
customers.merge(orders, on="customer_id", how="left").groupby("customer_id")["amount"].agg(["sum", "count"]).sort_values(by=["sum"] , ascending=False).fillna(0).head()  # Read, read, read. Missed count, and fillna(0)

,sum,count
customer_id,,
C00501,1776.98,10
C00530,1700.67,3
C04931,1666.24,6
C00837,1526.27,5
C02346,1464.93,11


---
## Part 2: SQL (10 min)

Write SQL queries for the following. Assume tables: `customers`, `orders`, `surveys` with the same columns as the CSVs.

### Q2.1
Find the **top 3 cities** by average `monthly_spend`, but only include cities with at least 100 customers.

In [87]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       5000 non-null   str    
 1   city              4766 non-null   str    
 2   gender            4375 non-null   str    
 3   age               4985 non-null   float64
 4   annual_income     4800 non-null   float64
 5   signup_date       4988 non-null   str    
 6   plan_type         4454 non-null   str    
 7   tenure_months     5000 non-null   int64  
 8   support_tickets   5000 non-null   int64  
 9   monthly_spend     4920 non-null   float64
 10  churned           5000 non-null   int64  
 11  customer_segment  5000 non-null   str    
dtypes: float64(3), int64(3), str(6)
memory usage: 468.9 KB


In [88]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     15000 non-null  str    
 1   customer_id  15000 non-null  str    
 2   order_date   15000 non-null  str    
 3   amount       14700 non-null  float64
 4   category     15000 non-null  str    
dtypes: float64(1), str(4)
memory usage: 586.1 KB


In [89]:
surveys.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         3000 non-null   str    
 1   satisfaction_score  2855 non-null   float64
 2   nps_score           2900 non-null   float64
 3   survey_date         3000 non-null   str    
dtypes: float64(2), str(2)
memory usage: 93.9 KB


In [100]:
# Write your SQL as a string (we'll validate it with sqlite)
q1 = """
SELECT city, AVG(monthly_spend)
FROM customers
GROUP BY city
HAVING COUNT(DISTINCT customer_id) > 100
ORDER BY AVG(monthly_spend) DESC
LIMIT 3;
"""


### Q2.2
For each customer, find their **most recent order date** and the **total amount** they've spent. Include customers who have never ordered (show NULLs for order fields).

Sort by total amount descending.

In [101]:
q2 = """
SELECT c.customer_id, MAX(o.order_date), SUM(o.amount)
FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id;  -- I know I'm missing the "Show NULLs for order fields", and that it probably has to do on how I'm aggregating data.
"""


### Q2.3
Find customers who have a **satisfaction score below 3** (from surveys) AND have **churned**. Return their `customer_id`, `city`, `satisfaction_score`, and `nps_score`.

In [102]:
q3 = """
SELECT c.customer_id, c.city, s.satisfaction_score, s.nps_score
FROM customers c
    INNER JOIN surveys s ON c.customer_id = s.customer_id
WHERE 
    c.churned = 1 
    AND s.satisfaction_score < 3;
"""


### Validate your SQL
Run this cell to test your queries against the actual data:

In [103]:
import sqlite3

conn = sqlite3.connect(':memory:')
customers.to_sql('customers', conn, index=False)
orders.to_sql('orders', conn, index=False)
surveys.to_sql('surveys', conn, index=False)

for label, query in [('Q2.1', q1), ('Q2.2', q2), ('Q2.3', q3)]:
    try:
        result = pd.read_sql(query, conn)
        print(f"\n{'='*40}")
        print(f"{label} — {len(result)} rows")
        print(result.head())
    except Exception as e:
        print(f"\n{label} — ERROR: {e}")


Q2.1 — 3 rows
          city  AVG(monthly_spend)
0  Chihuahua          108.647717
1  Guadalajara          108.401021
2    Querétaro          108.311935

Q2.2 — 4950 rows
  customer_id MAX(o.order_date)  SUM(o.amount)
0      C00001        2024-12-19         107.63
1      C00002        2024-09-02          52.52
2      C00003        2024-02-28          38.92
3      C00004        2023-05-07         224.99
4      C00005        2024-06-06         163.59

Q2.3 — 26 rows
  customer_id         city  satisfaction_score  nps_score
0      C00451  Guadalajara                 2.0        2.0
1      C00480       Mérida                 2.0        8.0
2      C00679  Chihuahua                 1.0        3.0
3      C00786       Puebla                 2.0       10.0
4      C00787         León                 2.0        7.0


---
## Part 3: sklearn & ML (10 min)

### Q3.1
Prepare the data for a churn prediction model:
1. Start from `customers` — select features: `age`, `annual_income`, `tenure_months`, `support_tickets`, `monthly_spend`
2. Drop rows with any NaN in these columns
3. Target = `churned`
4. Split into train/test (80/20, `random_state=42`)
5. Apply `StandardScaler` to the features (fit on train, transform both)

In [ ]:
# Your code here


### Q3.2
Train a `LogisticRegression` model on the scaled training data. Then:
1. Print the **accuracy** on the test set
2. Print the **classification report** (precision, recall, f1 for each class)
3. Which metric matters more here — **precision or recall** for the churn class? Why? (Answer in a comment)

In [ ]:
# Your code here

# Which metric matters more for churn prediction? Why?
# Answer: 


### Q3.3 (Bonus — if you have time)
Compute the **confusion matrix** and **ROC-AUC score** for your model. Plot the confusion matrix using matplotlib (a simple heatmap is fine).

In [ ]:
# Your code here


---
## ⏱️ Time's Up!

Stop here. Record how far you got:
- [ ] Part 1 complete
- [ ] Part 2 complete
- [ ] Part 3 complete
- [ ] Bonus complete

**Send your results to your reviewer.** We'll review together.